# Coherent Control of a Spin 1/2 via pulsed Electron Spin Resonance

In [ ]:
# Relevant Imports
import matplotlib.pyplot as plt
import numpy as np
from scipy import constants as const
from qutip import (
    tensor, qeye, sigmax, sigmay, sigmaz,
    basis, mesolve, sigmam, Qobj, Bloch
)

# Import the Simulation packages
from spinfinity import *
from spinfinity import helperfunctions as hf

## Setting up the static spin Hamiltonian

In [ ]:
spin = SpinSys([0.5]) # Initialize a single spin-1/2 system

# Static Hamiltonian parameters
spin.B = [0, 0, 0.5]
spin.BTip = [1e-9, 0, 0]

# Tunnel-junction parameters
# the tunnel junction parameters determine the inelastic scattering rates and therefore the T1 time
spin.T = 0.2
spin.U = [0]
spin.V_DC = 0
spin.G_ss = [1e-9]
spin.G_st = 1e-10

hamil.plot_ZeemanDiagramm(spin, energyscale='GHz');


### Cw-ESR frequency sweep

In [ ]:
ESR.plot_FreqSweep(spin, freq_array = np.linspace(12,16,1000), lw=0.05);

## Rabi Oscillations

### Building collapse operators

In [ ]:
# Relaxation
bath.calc_Rates(spin)
bath.calc_Lifetimes(spin)
T1_est = (1/spin.T_state_tot[0]+1/spin.T_state_tot[1])**-1*1e9
print(f"Estimated T1 time: {T1_est:.2f} ns")

c_ops_T1 = lind.calc_collapseOperators(spin, threshold_percent=1e-9, recalcRates=True)


# Decoherence
T_phi = 200  # ns

# Collapse operators needed to descibe the decoherence
gamma_phi = 1.0 / T_phi
Sz = 0.5 * sigmaz()
c_ops_Tphi = [np.sqrt(2*gamma_phi) * Sz]


# Combine the collapse operators for relaxation and decoherence
c_ops = c_ops_T1 + c_ops_Tphi
spin.collapse_ops = c_ops


## Preparing pulse parameters

In [ ]:
# Calculating the larmor frequency of the electron
B_tot = np.linalg.norm(np.array(spin.B) + np.array(spin.BTip))
mu_B = const.physical_constants['Bohr magneton'][0]
f_e = 2.0 * B_tot / const.h * 1e-9 * mu_B   # GHz

# Setting the RF frequency and Rabi rate
detuning = -0.00e-3 #GHz
f_RF = f_e + detuning # GHz
RabiRate_GHz = 50*1e-3 # GHz

# Assigning the RF frequency and Rabi rate to the spin system
spin.f_RF = f_RF
spin.RabiRate = RabiRate_GHz

## Single Rabi Oscillation

In [ ]:
# Initial state 
# For the Rabi oscillation simulation, the system needs to be prepared in an initial state.
# Here, we prepare the system in the thermal state at the given temperature.
rho_therm = lind.initialize_thermal(spin)

# Rabi Oscillation Simulation
# The Rabi oscillation simulation is performed by varying the pulse duration (tau) 
# and calculating the final expectation value of S_x, S_y, S_z after the pulse.
tau_array = np.linspace(0, 200, 2000) # ns
_, result = CC.calc_RabiOscillation(spin, tau_array, e_ops=None) # if e_ops is None, the expectation values of S_x, S_y, S_z are calculated.


# Perform Oscillation Fit
x = result.times
y = result.expect[2]
x_fit, y_fit, popt, perr = hf.fit_oscillation(x, y)
A_fit, T_fit, RabiRate_fit, phi_fit, C_fit = popt


# Plot
plt.figure(figsize=(10, 5))
plt.plot(tau_array, result.expect[2], "o", ms=5, label="Simulation Data")
plt.plot(x_fit, y_fit, "-", lw=2, label=f"Fit: A={A_fit:.3f}, T={T_fit:.1f} ns, RabiRate={RabiRate_fit:.3f} GHz, phi={phi_fit:.2f}, C={C_fit:.3f}")
plt.plot()
plt.xlabel(r"Pulse Duration $\tau$ (ns)")
plt.ylabel(r"Final $\langle S_{z} \rangle$")
plt.legend()
plt.ylim([-0.5,0.5])
plt.tight_layout()
plt.show()

# Rabi Decoherence time 
T2_Rabi = 1/(1/(2*T_phi) + 3/(4*T1_est))
print(f"Estimated Rabi decoherence time: {T2_Rabi:.2f} ns")


## Representation on the Bloch Sphere

In [ ]:
sphere = Bloch()
sphere.add_points(
[
    2 * result.expect[0],
    2 * result.expect[1],
    2 * result.expect[2],
],
meth="l",
)
sphere.show()

## Detuning Experiment

In [ ]:
freq_array = np.linspace(f_e-50e-3,f_e+50e-3,200)
_,_,signal_map = CC.calc_RabiDetuningMap(spin, tau_array, freq_array)

plt.figure(figsize=(8, 5))
extent = [
    tau_array[0], tau_array[-1],
    (freq_array[0]-f_e) * 1e3, (freq_array[-1]-f_e) * 1e3
]

plt.imshow(
    signal_map,
    aspect='auto',
    origin='lower',
    extent=extent,
    interpolation='none',
    cmap='magma'
)

plt.xlabel(r'Pulse Width $\tau$ (ns)')
plt.ylabel(r'Detuning $\delta = f_{\mathrm{RF}} - f_e$ (MHz)')
plt.clim([-0.5,0.5])
plt.colorbar(label=r'Final $\langle S_{z} \rangle$')
plt.tight_layout()
plt.show()

# Relaxometry

In [ ]:
tau_array = np.linspace(0, 5*T1_est, 1000) # ns
spin.f_RF = f_e
_ = lind.initialize_thermal(spin)

_, result_relax = CC.calc_Relaxometry(spin, tau_array, e_ops=None)

# Perform an exponential Fit
x = result_relax.times
y = result_relax.expect[2]
x_fit, y_fit, popt, perr = hf.fit_exponential(x, y)
A_fit, T_fit, C_fit = popt

# Plot the result
plt.figure(figsize=(7, 4))
plt.plot(x, y, "o", ms=3, label="Simulated trace")
plt.plot(x_fit, y_fit, "-", lw=2, label="Exponential fit")
plt.xlabel(r"Free evolution time $\tau$ (ns)")
plt.ylabel(r"Final $\langle S_z \rangle$")
plt.title(f"Fit: A={A_fit:.4f}, T={T_fit:.2f} ns, C={C_fit:.4f}")
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

print(f"Estimated T1 time tunnel rates: {T1_est:.2f} ns")

## Ramsey 

In [ ]:
# Plot a Ramsey Shot
detuning = 50.00e-3 #GHz
spin.f_RF = f_e + detuning
_ = lind.initialize_thermal(spin)

CC.plot_RamseyShot(spin, tau=30);

In [ ]:
tau_array = np.linspace(0, 200, 1000) # ns

_, Ramsey_signal = CC.calc_Ramsey_Parallel(spin, tau_array, e_ops=None)

# Perform Oscillation Fit
x = tau_array
y = Ramsey_signal[:,2]
x_fit, y_fit, popt, perr = hf.fit_oscillation(x, y)
A_fit, T_fit, RabiRate_fit, phi_fit, C_fit = popt


# Plot
plt.figure(figsize=(10, 5))
plt.plot(x, y, "o", ms=5, label="Simulation Data")
plt.plot(x_fit, y_fit, "-", lw=2, label=f"Fit: A={A_fit:.3f}, T={T_fit:.1f} ns, RabiRate={RabiRate_fit:.3f} GHz, phi={phi_fit:.2f}, C={C_fit:.3f}")
plt.plot()
plt.xlabel(r"Pulse Duration $\tau$ (ns)")
plt.ylabel(r"Final $\langle S_{z} \rangle$")
plt.legend()
plt.ylim([-0.5,0.5])
plt.tight_layout()
plt.show()

## Ramsey Detuning Map

In [ ]:
freq_array = np.linspace(f_e-50e-3,f_e+50e-3,200)
tau_array = np.linspace(0, 200, 400) # ns
_,_,Ramsey_map = CC.calc_RamseyDetuningMap(spin, tau_array, freq_array)

plt.figure(figsize=(8, 5))
extent = [
    tau_array[0], tau_array[-1],
    (freq_array[0]-f_e) * 1e3, (freq_array[-1]-f_e) * 1e3
]

plt.imshow(
    Ramsey_map,
    aspect='auto',
    origin='lower',
    extent=extent,
    interpolation='none',
    cmap='magma'
)

plt.xlabel(r'Waiting time $\tau$ (ns)')
plt.ylabel(r'Detuning $\delta = f_{\mathrm{RF}} - f_e$ (MHz)')
plt.clim([-0.5,0.5])
plt.colorbar(label=r'Final $\langle S_{z} \rangle$')
plt.tight_layout()
plt.show()

## Hahn Echo

In [ ]:
THE_est = (1/(1/(T_phi) + 1/(2*T1_est)))
tau_array = np.linspace(0, 5*THE_est, 1000) # ns
spin.f_RF = f_e
_ = lind.initialize_thermal(spin)

_, HE_signal = CC.calc_HahnEcho_pluspihalf_Parallel(spin, tau_array, e_ops=None)

# Perform an exponential Fit
x = tau_array * 2
y = HE_signal[:,2]
x_fit, y_fit, popt, perr = hf.fit_exponential(x, y)
A_fit, T_fit, C_fit = popt

# Plot the result
plt.figure(figsize=(7, 4))
plt.plot(x, y, "o", ms=3, label="Simulated trace")
plt.plot(x_fit, y_fit, "-", lw=2, label="Exponential fit")
plt.xlabel(r"Total free evolution time $\tau$ (ns)")
plt.ylabel(r"Final $\langle S_z \rangle$")
plt.title(f"Fit: A={A_fit:.4f}, T={T_fit:.2f} ns, C={C_fit:.4f}")
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

print(f"Estimated T2 time: {THE_est:.2f} ns")